# Intro to ML and Linear Regression (Simplified & Enhanced)

This notebook is based on the [Google ML Crash Course](https://developers.google.com/machine-learning/crash-course). We'll learn how to predict taxi fares using math and code.

---

## 🚗 The Problem: How much for a ride?

We want to predict the **Fare** (Price) of a taxi ride. 

- **Label (Y)**: What we want to predict (e.g., Fare).
- **Feature (X)**: What we use to make the prediction (e.g., Trip Miles).

In a simple world, the relationship is a straight line:  
**Fare = (Rate × Miles) + Base Fee**

In ML terms, this is:  
**y = wx + b**

---

## 📉 Understanding Loss (How wrong are we?)

**Loss** is a number that tells us how far off our prediction was from the actual price. 

### 1. Mean Squared Error (MSE)
This is the most common loss function. It squares the error (Difference between actual and predicted).

**The Logic:** If the error is 2, the loss is 4. If the error is 10, the loss is 100!  
**Why?** It heavily punishes the model for big mistakes.

**Formula:**
$$Loss = \frac{(Actual - Predicted)^2}{Count}$$

### 2. Mean Absolute Error (MAE)
This simply takes the absolute difference (ignores whether the error was positive or negative).

**The Logic:** If the error is 2, the loss is 2. If the error is 10, the loss is 10.  
**Why?** It treats all errors proportionally. It's "gentler" than MSE.

---

## 🔄 Gradient Descent: The "Downhill" Algorithm

Imagine you are at the top of a foggy mountain and want to reach the lowest valley (where the **Loss** is zero). 

1. **Look around**: Feel which way the ground slopes down (Calculate the **Gradient**).
2. **Take a step**: Move a small distance in that downward direction (The **Learning Rate** controls how big the step is).
3. **Repeat**: Keep doing this until you reach the bottom (Convergence).

### 👨‍💻 Manual Gradient Descent Example
Here is how a computer actually updates the "Weight" (Rate) to learn from its mistakes:

In [ ]:
# A simple demonstration of how weights are updated manually
weight = 5.0      # Starting guess for Rate per mile
learning_rate = 0.1

actual_fare = 25.0
miles = 2.0
predicted_fare = weight * miles  # Current prediction: 10.0

print(f"Initial Prediction: ${predicted_fare}")

# 1. Calculate Error
error = predicted_fare - actual_fare  # 10 - 25 = -15

# 2. Calculate Gradient (simplified derivative of squared error)
gradient = 2 * error * miles

# 3. Update Weight (The learning part!)
weight = weight - (learning_rate * gradient)

new_prediction = weight * miles
print(f"New Weight: {weight}")
print(f"New Prediction after 1 update: ${new_prediction} (Much closer to 25!)")

---

## ⚙️ Hyperparameters (The Control Knobs)

- 🎓 **Learning Rate**: Step size. 
    - Too big? You might jump over the valley.
    - Too small? It will take years to reach the bottom.
- 🔄 **Epochs**: How many times the model looks at the entire dataset.
- 📦 **Batch Size**: How many examples to look at before taking a single step.

In [ ]:
# ──────────────────────────────────────────────────────
# Real Implementation with TensorFlow
# ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Load data
DATA_URL = 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv'
df = pd.read_csv(DATA_URL)
training_df = df.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE')].dropna()

def build_and_train(df, feature_names, lr=0.001, epochs=20):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=1, input_shape=(len(feature_names),))
    ])
    model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=lr), 
                  loss='mean_squared_error', 
                  metrics=[tf.keras.metrics.RootMeanSquaredError()])
    
    history = model.fit(x=df[feature_names], y=df['FARE'], 
                        batch_size=50, epochs=epochs, verbose=0)
    return model, history

# Run Experiment
model, history = build_and_train(training_df, ['TRIP_MILES'])

plt.plot(history.history['root_mean_squared_error'])
plt.title('Model Learning (RMSE decreasing over time)')
plt.ylabel('Error ($)')
plt.xlabel('Epoch')
plt.show()